> **Public evidence copy:** Stored outputs and execution counts have been removed. 
> The participant-level workbook is not included. Set `JAI_SURVEY_DATA` to an 
> authorized secure copy before running this notebook.


# Dog ownership and household dog movement

## Setup

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px 
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import os 

In [ ]:
from pathlib import Path

data_path_value = os.environ.get("JAI_SURVEY_DATA")
if not data_path_value:
    raise RuntimeError(
        "Set JAI_SURVEY_DATA to the absolute path of the restricted survey workbook. "
        "Do not copy participant data into this repository."
    )

file = Path(data_path_value).expanduser().resolve()
if not file.is_file():
    raise FileNotFoundError(f"Restricted survey workbook not found: {file}")



In [ ]:
df1 = pd.read_excel(file, 'Jai KAP Survey 2025')
df3 = pd.read_excel(file, 'dog_details_current')


In [ ]:
df1_filtered = df1[
    (df1['Is someone >18 present at the household?'] == 'Yes') &
    (df1['Has the respondent consented to undertake the survey?'] == 'Yes')
]



In [ ]:
df1_ong = df1_filtered[df1_filtered["District"] == "Ongkharak"]
df1_pak = df1_filtered[df1_filtered["District"] == "Pak Phli"]


> Records without consent or without an eligible adult respondent are excluded according to the analysis cleaning rule.


## Analysis


### Q24 – Do you currently own any dogs?

In [ ]:
# Distribution of dog ownership responses by district

for name, df in [("Pak Phli", df1_pak), ("Ongkharak", df1_ong)]:
    print(f"{name}:\n{df.iloc[:, 20].value_counts().sort_index()}\n")

In [ ]:
# Distribution of the number of dogs owned by district


for name, df in [("Pak Phli", df1_pak), ("Ongkharak", df1_ong)]:
    print(f"{name}:\n{df.iloc[:, 21].value_counts().sort_index()}\n")

In [ ]:
for name, df in [("Pak Phli", df1_pak), ("Ongkharak", df1_ong)]:
    print(f"{name}:\n{df.iloc[:, 21].describe()}\n")

In [ ]:
col = "How many dogs do you currently own?"

fig = px.box(df1_filtered, x="District", y=col,
             color="District",
             color_discrete_map={"Pak Phli":"blue","Ongkharak":"#d16443"},
             category_orders={"District":["Pak Phli","Ongkharak"]})
fig.update_layout(title=col, yaxis_title="Number of dogs", width=800, height=500)
fig.show()

df1_filtered.groupby("District")[col].mean()


In [ ]:
df.columns[18:20]

In [ ]:
# Q17: % of households where dogs left
col_leave_yes = "In the past 12 months, did any dogs leave your household?"
leave_rate = df1_filtered.groupby("District")[col_leave_yes].value_counts(normalize=True).unstack(fill_value=0) * 100
print(leave_rate)

print('------------------')
# Q18: average number of dogs that left
col_leave_count = "How many dogs have left your household in the last 12 months?"
avg_left = df1_filtered.groupby("District")[col_leave_count].mean()
print(avg_left)


In [ ]:
# Columns
col_leave_yes   = "In the past 12 months, did any dogs leave your household?"
col_leave_count = "How many dogs have left your household in the last 12 months?"

# --- Consistent settings (match your previous donuts) ---
district_order = ["Pak Phli", "Ongkharak"]
resp_order     = ["Yes", "No"]
color_map      = {"Yes": "#54C788", "No": "#d75b4f"}  # green/red

# =========================
# FIGURE 1 — DONUT (Yes/No)
# =========================

# Keep only valid Yes/No responses
df_yesno = df1_filtered.loc[df1_filtered[col_leave_yes].isin(resp_order), ["District", col_leave_yes]]

# Counts per district (Yes/No)
counts_tbl = (
    df_yesno.groupby("District")[col_leave_yes]
    .value_counts()
    .unstack(fill_value=0)
    .reindex(index=district_order)
    .reindex(columns=resp_order, fill_value=0)
)

# Prepare subplot with two donuts (one per district)
fig1 = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "pie"}, {"type": "pie"}]],
    subplot_titles=("Pak Phli (CNVR)", "Ongkharak (Non-CNVR)")
)

# Add pies
for i, d in enumerate(district_order, start=1):
    vals = counts_tbl.loc[d]
    total = vals.sum()
    fig1.add_trace(
        go.Pie(
            labels=vals.index,
            values=vals.values,
            hole=0.6,  # match previous donut width
            marker=dict(colors=[color_map[k] for k in vals.index]),
            textinfo="percent+label",
            name=d,
            hovertemplate="%{label}: %{value} households (%{percent})<extra></extra>"
        ),
        row=1, col=i
    )

# Annotations above each donut (same positioning as before)
annotations = []
for i, d in enumerate(district_order):
    annotations.append(dict(
        text=f"<b>{d}</b>",
        x=0.225 if i == 0 else 0.775,
        y=0.5,
        showarrow=False,
        font=dict(size=16)
    ))

fig1.update_layout(
    title_text="Households Reporting Dogs Leaving (Past 12 Months)",
    title_x=0.5,
    height=650,
    width=1200,
    legend_title_text="Response",
    annotations=annotations
)

# Ensure consistent hole size across traces
fig1.update_traces(hole=0.6, selector=dict(type="pie"))

fig1.show()

# ==============================
# FIGURE 2 — BAR (Average Count)
# ==============================

# Average number of dogs that left (coerce numeric)
avg_left = (
    pd.to_numeric(df1_filtered[col_leave_count], errors="coerce")
    .groupby(df1_filtered["District"])
    .mean()
).reindex(district_order)

fig2 = go.Figure()

fig2.add_trace(
    go.Bar(
        x=avg_left.index,
        y=avg_left.values,
        name="Average dogs left",
        marker_color="#636efa",
        text=avg_left.round(2).astype(str),
        textposition="outside",
        hovertemplate="<b>%{x}</b><br>Average: %{y:.2f}<extra></extra>",
    )
)

fig2.update_layout(
    title_text="Average Number of Dogs That Left (Past 12 Months)",
    height=500,
    width=800,
    xaxis_title="District",
    yaxis_title="Average dogs left",
    margin=dict(l=60, r=40, t=80, b=60),
)

fig2.update_xaxes(categoryorder="array", categoryarray=district_order)
fig2.update_yaxes(rangemode="tozero")

fig2.show()


In [ ]:
# merge df3._parent_index with df1._index 

In [ ]:
# --- 1) Merge District into df3 via parent/child keys ---
# Ensure keys are same dtype
df1_filtered["_index"] = df1_filtered["_index"].astype(str)
df3["_parent_index"]   = df3["_parent_index"].astype(str)

# Keep only the columns we need from df1
keymap = df1_filtered[["_index", "District"]].drop_duplicates()

# Merge: each dog (df3) gets its household's District
df3m = df3.merge(keymap, left_on="_parent_index", right_on="_index", how="left")

# Optional: drop rows without a matched district
df3m = df3m.dropna(subset=["District"])

# --- 2) Q33–34: Where did you acquire this dog? (stacked %) ---
col = "Where did you acquire this dog?"

# If you want to tidy up NAs:
df3m = df3m[df3m[col].notna()]

acq_counts = df3m.groupby("District")[col].value_counts().unstack(fill_value=0)
acq_percent = acq_counts.div(acq_counts.sum(axis=1), axis=0) * 100

# Optional: enforce district order
acq_percent = acq_percent.reindex(index=["Pak Phli", "Ongkharak"])

# Plot
fig = go.Figure()
for source in acq_percent.columns:
    fig.add_trace(go.Bar(
        x=acq_percent.index,     # Districts
        y=acq_percent[source],   # %
        name=source
    ))

fig.update_layout(
    barmode="stack",
    title="Where Did You Acquire This Dog?",
    xaxis_title="District",
    yaxis_title="Percent of Owned Dogs",
    height=600,
    width=1200,
    legend_title_text="Source"
)
fig.show()


In [ ]:
acq_counts


In [ ]:
df3["Where did you acquire this dog?"].value_counts(dropna=False)
